# Signaling Analysis
### Analyzing the performance of the signaling apparatus

In [17]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

In [18]:
active_pairs = pd.read_csv("../data/manipulated/active_pairs.csv", parse_dates=True)

train_residuals = pd.read_csv("../data/manipulated/residuals_train.csv", index_col=0, parse_dates=True)

train_start = train_residuals.index.min()
train_end = train_residuals.index.max()

In [19]:
def compute_match_percentage(df):
    # exclude rows where signal == 0
    df = df[df["signal"] != 0]

    # define match condition
    matches = (
        ((df["signal"] == 1) & (df["minhash_signal"] > 0)) |
        ((df["signal"] == -1) & (df["minhash_signal"] < 0))
    )

    return matches.mean()  # proportion

In [20]:
train_results = []
full_results = []

for _, row in active_pairs.iterrows():
    a, b = row['asset_a'], row['asset_b']
    filename = f"../data/manipulated/spreads/{a}_{b}.csv"
    
    full = pd.read_csv(filename, index_col=0, parse_dates=True)
    train = full.loc[train_start:train_end]

    train_pct = compute_match_percentage(train)
    full_pct = compute_match_percentage(full)

    train_results.append(train_pct)
    full_results.append(full_pct)

# optional: summary stats
print("Train mean match %:", np.mean(train_results))
print("Full mean match %:", np.mean(full_results))

Train mean match %: 0.4362429417728824
Full mean match %: 0.44725703361588226


In [21]:
results = []

for _, row in active_pairs.iterrows():
    a, b = row['asset_a'], row['asset_b']
    filename = f"../data/manipulated/spreads/{a}_{b}.csv"
    
    full = pd.read_csv(filename, index_col=0, parse_dates=True)
    train = full.loc[train_start:train_end]

    results.append({
        "pair": f"{a}_{b}",
        "train_match_pct": compute_match_percentage(train),
        "full_match_pct": compute_match_percentage(full)
    })

results_df = pd.DataFrame(results)
print(results_df)

         pair  train_match_pct  full_match_pct
0     CDW_HAS         0.502825        0.502183
1    CTSH_CSX         0.406417        0.422222
2      HAS_EL         0.456522        0.480000
3    AMCR_ACN         0.431953        0.422330
4    AMCR_MSI         0.430769        0.431535
..        ...              ...             ...
135   APO_BLK         0.472603        0.490251
136   HAS_ACN         0.427746        0.458333
137   CRH_ACN         0.474747        0.484848
138   CRH_MSI         0.457627        0.448468
139  CBRE_MSI         0.467532        0.478261

[140 rows x 3 columns]


In [22]:
print(results_df["full_match_pct"].min())
print(results_df["full_match_pct"].max())

0.35
0.5517241379310345
